## Supplementary Table S3: quantified TMEM106B glycopeptiforms and their use in downstream analyses

In [ ]:
suppressMessages(library(SummarizedExperiment))

suppressMessages(library(SummarizedExperiment))

glyco <- readRDS("tmem106b/glyco/glyco_tmem106b.Rds")

In [ ]:
# Manuscript cohort: genotype + nonmissing tau (316 samples)
genotypes <- c("C/C", "C/G", "G/G")
keep <- colData(glyco)$rs3173615 %in% genotypes & !is.na(colData(glyco)$tangles_sqrt)
glyco <- glyco[, keep]
stopifnot(ncol(glyco) == 316)

# One row per glycopeptiform; derive annotations from the R object.
f <- as.data.frame(rowData(glyco))
mass <- as.numeric(sub(".*N\\[([0-9.]+)\\].*", "\\1", f$Modified.Sequence))
start <- f$Start
start[f$Peptide.Sequence == "LNNISIIGPLDMK"] <- unique(f$Start[f$Peptide.Sequence == "LNNITIIGPLDMK"])
site <- regmatches(f$Assigned.Modifications, regexpr("[0-9]+N\\(", f$Assigned.Modifications))
stopifnot(length(site) == nrow(f), !anyNA(mass))

In [ ]:
# The RDS has masses but no compositions; label the known study compositions.
masses <- c("HexNAc(2)Hex(5)" = 1216.4229, "HexNAc(2)Hex(6)" = 1378.4757, "HexNAc(2)Hex(7)" = 1540.5284,
            "HexNAc(2)Hex(8)" = 1702.5813, "HexNAc(3)Hex(4)Fuc(1)" = 1403.5073, "HexNAc(7)Hex(8)NeuAc(1)" = 3009.0735)
hit <- vapply(mass, function(m) { k <- which(abs(masses - m) < .001); if (length(k) == 1) k else NA_integer_ }, integer(1))
tableS3 <- data.frame(feature_id = rownames(glyco), sequence = f$Modified.Sequence, composition = names(masses)[hit],
                      glycosite = start + as.integer(sub("N\\(", "", site)) - 1L,
                      oxidized = grepl("M[15.9949]", f$Modified.Sequence, fixed = TRUE))

I <- assay(glyco, "intensity"); M <- assay(glyco, "match_type"); S <- assay(glyco, "spectral_count")
stopifnot(!anyNA(I), all(is.finite(I)), all(I >= 0))
# Same evidence definition for every row; no site-specific or paired-allele filter.
msms <- I > 0 & M == "MS/MS" & S > 0
mbr <- I > 0 & M == "MBR"
msms[is.na(msms)] <- mbr[is.na(mbr)] <- FALSE
tableS3$n_quantified <- rowSums(I > 0); tableS3$n_MSMS <- rowSums(msms); tableS3$n_MBR <- rowSums(mbr)
tableS3$Analysis <- "Not used"
tableS3$Analysis[tableS3$feature_id %in% paste("TMEM", c("GP02", "GP04", "GP08", "GP10", "GP14", "GP16"), sep = "_")] <- "Used"
# Analysis: "Used" = unoxidized Hex5-7 glycopeptiforms of the primary analyses; every other feature (oxidized, Hex8 exploratory, rare) is "Not used"

In [ ]:
n_omitted <- sum(tableS3$n_quantified == 0)
tableS3 <- tableS3[tableS3$n_quantified > 0, ]
colnames(tableS3)[1:5] <- c("ID", "Sequence", "Composition", "Glycosite", "Oxidized")

# Footer note row (first column starts with "Note:", all other columns empty).
note <- paste0("Note: Bracketed values in Sequence indicate modification masses (Da). n_quantified counts samples with positive intensity, ",
  "partitioned into direct MS/MS-supported (n_MSMS) and match-between-runs (n_MBR) observations.")
out <- rbind(tableS3, tableS3[NA_integer_, ][1, ])
out$ID <- as.character(out$ID); out$ID[nrow(out)] <- note
write.csv(out, "TableS3.csv", row.names = FALSE, na = "")
cat("TableS3.csv:", nrow(tableS3), "data rows;", n_omitted, "omitted\n")